# Stage 9.4 — Offline Live-Style Video Recognition Validation

## Purpose

Stage 9.4 converts the validated Stage 9.3 OpenVINO retrieval pipeline into an
offline deployment-style recognizer.

Unlike Stage 9.3, this stage does not use historical Stage 4.5 `sample_indices`
to determine which frames should be processed. Instead, the recognizer accepts
a video path directly, determines the available video frames, automatically
selects an 8-frame temporal sample across the clip, extracts OpenVINO spatial
embeddings, performs mean temporal pooling, and retrieves the Top-k CISLR
prototype glosses.

The purpose of this stage is to bridge the controlled research evaluation
pipeline and the upcoming real-time webcam pipeline.

---

## Frozen recognition pipeline

Video  
→ automatic 8-frame temporal sampling  
→ full-frame preprocessing  
→ OpenVINO MobileNetV3-Large  
→ 960-D frame embeddings  
→ mean temporal pooling  
→ L2 normalization  
→ cosine similarity  
→ frozen Stage 4.5 prototype gallery  
→ Top-5 gloss predictions

---

## Scientific / Methodological Disclaimer

Stage 9.4 is a deployment validation stage and does not perform model training,
fine-tuning, prototype adaptation, test-label tuning, Hand-ROI processing,
whitening, CSLS, or test-set optimization.

The OpenVINO embedding extractor and Stage 4.5 prototype gallery are frozen.

The test video labels are used only after inference for reporting whether the
retrieved gloss agrees with the known CISLR annotation. They are not supplied
to the recognition algorithm.

Stage 9.4 uses complete isolated-sign video clips. Automatic uniform sampling
over the complete clip is therefore possible. This should not be confused with
the upcoming webcam system, where the end of a sign is not known in advance
and a rolling temporal buffer will be required.

The output of this stage represents isolated-sign prototype retrieval and must
not be described as continuous sign-language translation.

---

## Success Criteria

Stage 9.4 passes when:

1. An arbitrary CISLR video can be supplied without historical sample indices.
2. Eight temporal frames are selected automatically.
3. All selected frames are processed by the frozen OpenVINO model.
4. A normalized 960-D video representation is generated.
5. Top-5 prototype glosses are returned.
6. Processing latency is measured.
7. Predictions and diagnostic information are saved.
8. No upstream research artifacts are modified.

Successful completion permits progression to Stage 9.5 live webcam inference.

In [1]:
# ==================================================================================================
# CISLR — STAGE 9.4
# OFFLINE LIVE-STYLE VIDEO RECOGNITION VALIDATION
# ==================================================================================================
#
# INPUT
# -----
# Arbitrary prerecorded CISLR isolated-sign video
#
# PIPELINE
# --------
# Video
#   -> automatic uniform 8-frame sampling
#   -> full-frame preprocessing
#   -> OpenVINO MobileNetV3-Large
#   -> 8 x 960-D frame embeddings
#   -> mean temporal pooling
#   -> L2 normalization
#   -> cosine similarity
#   -> canonical Stage 4.5 prototype gallery
#   -> Top-5 glosses
#
# IMPORTANT
# ---------
# NO TRAINING
# NO FINE-TUNING
# NO HISTORICAL sample_indices
# NO HAND ROI
# NO TTA
# NO WHITENING
# NO CSLS
# NO TEST-LABEL TUNING
#
# ==================================================================================================

from pathlib import Path
import json
import time

import cv2
import numpy as np
import pandas as pd

from openvino import Core


# ==================================================================================================
# 0. CONFIGURATION
# ==================================================================================================

ROOT = Path("/home/dipshikha/Music/cao/CISLR_RESEARCH")

VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

STAGE9_ROOT = (
    ROOT
    / "audit"
    / "stage9_final_openvino"
)

STAGE9_4_DIR = (
    STAGE9_ROOT
    / "offline_validation"
)

REPORT_DIR = (
    STAGE9_4_DIR
    / "reports"
)

PREDICTION_DIR = (
    STAGE9_4_DIR
    / "predictions"
)

STAGE9_4_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# Frozen Stage 9.1 OpenVINO embedding model
# --------------------------------------------------------------------------------------------------

OV_XML = (
    STAGE9_ROOT
    / "embedding_fp32"
    / "mobilenet_v3_large_embedding_960_fp32_static.xml"
)


# --------------------------------------------------------------------------------------------------
# Frozen canonical Stage 4.5 prototype gallery
# --------------------------------------------------------------------------------------------------

STAGE45_EMBED_DIR = (
    ROOT
    / "audit"
    / "stage4_5_official_evaluation"
    / "embeddings"
)

PROTOTYPE_EMBED_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_embeddings.npy"
)

PROTOTYPE_META_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_metadata.csv"
)

TEST_META_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_test_metadata.csv"
)


# --------------------------------------------------------------------------------------------------
# Deployment configuration
# --------------------------------------------------------------------------------------------------

TEMPORAL_FRAMES = 8
TOP_K = 5

# Use CPU first because it is already validated.
OPENVINO_DEVICE = "CPU"


print("=" * 100)
print("CISLR — STAGE 9.4")
print("OFFLINE LIVE-STYLE VIDEO RECOGNITION VALIDATION")
print("=" * 100)


# ==================================================================================================
# 1. ARTIFACT CHECK
# ==================================================================================================

required_files = {
    "OpenVINO model":
        OV_XML,

    "Prototype embeddings":
        PROTOTYPE_EMBED_PATH,

    "Prototype metadata":
        PROTOTYPE_META_PATH,

    "Test metadata":
        TEST_META_PATH,
}


print("\nARTIFACT CHECK")
print("-" * 100)


for name, path in required_files.items():

    print(
        f"{name:25s}: {path}"
    )

    print(
        f"{'':25s}  Exists = {path.exists()}"
    )


missing = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]


if missing:

    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(missing)
    )


print("\n[OK] All required artifacts exist.")


# ==================================================================================================
# 2. LOAD PROTOTYPE GALLERY
# ==================================================================================================

prototype_bank = np.load(
    PROTOTYPE_EMBED_PATH
).astype(np.float32)

prototype_meta = pd.read_csv(
    PROTOTYPE_META_PATH
)

test_meta = pd.read_csv(
    TEST_META_PATH
)


if prototype_bank.ndim != 2:

    raise ValueError(
        f"Expected 2-D prototype bank. "
        f"Got {prototype_bank.shape}"
    )


if prototype_bank.shape[1] != 960:

    raise ValueError(
        f"Expected 960-D prototype embeddings. "
        f"Got {prototype_bank.shape[1]}"
    )


if len(prototype_bank) != len(prototype_meta):

    raise ValueError(
        "Prototype bank and metadata are not aligned: "
        f"{len(prototype_bank)} vs "
        f"{len(prototype_meta)}"
    )


if "uid" not in prototype_meta.columns:

    raise ValueError(
        "Prototype metadata has no uid column."
    )


if "gloss" not in prototype_meta.columns:

    raise ValueError(
        "Prototype metadata has no gloss column."
    )


# --------------------------------------------------------------------------------------------------
# Normalize prototype gallery
# --------------------------------------------------------------------------------------------------

prototype_norms = np.linalg.norm(
    prototype_bank,
    axis=1,
    keepdims=True
)

prototype_bank = (
    prototype_bank
    / np.clip(
        prototype_norms,
        1e-12,
        None
    )
).astype(np.float32)


print("\n" + "=" * 100)
print("PROTOTYPE GALLERY")
print("=" * 100)

print(
    "Prototype count :",
    len(prototype_bank)
)

print(
    "Embedding dim   :",
    prototype_bank.shape[1]
)

print(
    "Gloss count     :",
    prototype_meta["gloss"].nunique()
)

print("\n[OK] Prototype gallery ready.")


# ==================================================================================================
# 3. LOAD OPENVINO MODEL
# ==================================================================================================

core = Core()

ov_model = core.read_model(
    str(OV_XML)
)

compiled_model = core.compile_model(
    ov_model,
    OPENVINO_DEVICE
)

input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)


print("\n" + "=" * 100)
print("OPENVINO MODEL")
print("=" * 100)

print(
    "Device      :",
    OPENVINO_DEVICE
)

print(
    "Input shape :",
    tuple(input_layer.shape)
)

print(
    "Output shape:",
    tuple(output_layer.shape)
)


if tuple(input_layer.shape) != (
    1,
    3,
    224,
    224
):

    raise ValueError(
        "Unexpected OpenVINO input shape: "
        f"{tuple(input_layer.shape)}"
    )


if tuple(output_layer.shape) != (
    1,
    960
):

    raise ValueError(
        "Unexpected OpenVINO output shape: "
        f"{tuple(output_layer.shape)}"
    )


print("\n[OK] OpenVINO model ready.")


# ==================================================================================================
# 4. PREPROCESSING
# ==================================================================================================

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(
    1,
    1,
    3
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(
    1,
    1,
    3
)


def preprocess_frame(frame_bgr):
    """
    Frozen preprocessing validated during Stage 9.2 and Stage 9.3.
    """

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    frame_rgb = cv2.resize(
        frame_rgb,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )

    x = (
        frame_rgb.astype(np.float32)
        / 255.0
    )

    x = (
        x - IMAGENET_MEAN
    ) / IMAGENET_STD

    x = np.transpose(
        x,
        (2, 0, 1)
    )

    x = np.expand_dims(
        x,
        axis=0
    )

    return np.ascontiguousarray(
        x,
        dtype=np.float32
    )


# ==================================================================================================
# 5. VIDEO INFORMATION
# ==================================================================================================

def inspect_video(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    cap.release()


    duration_seconds = (
        frame_count / fps
        if fps > 0
        else None
    )


    return {
        "frame_count":
            frame_count,

        "fps":
            fps,

        "width":
            width,

        "height":
            height,

        "duration_seconds":
            duration_seconds,
    }


# ==================================================================================================
# 6. AUTOMATIC TEMPORAL SAMPLING
# ==================================================================================================

def generate_uniform_indices(
    frame_count,
    num_frames=8
):
    """
    Automatically choose num_frames indices over the entire video.

    No historical sample_indices are used.
    """

    if frame_count <= 0:

        raise ValueError(
            f"Invalid frame count: {frame_count}"
        )


    if frame_count >= num_frames:

        indices = np.linspace(
            0,
            frame_count - 1,
            num_frames
        )

        indices = np.rint(
            indices
        ).astype(int)


    else:

        # Very short video:
        # repeat available positions to maintain exactly 8 frames.

        indices = np.linspace(
            0,
            frame_count - 1,
            num_frames
        )

        indices = np.rint(
            indices
        ).astype(int)


    indices = np.clip(
        indices,
        0,
        frame_count - 1
    )


    return indices.tolist()


# ==================================================================================================
# 7. READ AUTOMATICALLY SELECTED FRAMES
# ==================================================================================================

def read_sampled_frames(
    video_path,
    frame_indices
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )


    frames = []


    try:

        for frame_idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_idx)
            )

            ok, frame = cap.read()


            if not ok or frame is None:

                raise RuntimeError(
                    f"Could not decode frame "
                    f"{frame_idx} from "
                    f"{video_path}"
                )


            frames.append(frame)


    finally:

        cap.release()


    if len(frames) != TEMPORAL_FRAMES:

        raise RuntimeError(
            f"Expected {TEMPORAL_FRAMES} frames, "
            f"decoded {len(frames)}."
        )


    return frames


# ==================================================================================================
# 8. OPENVINO FRAME EMBEDDING
# ==================================================================================================

def extract_frame_embedding(
    frame_bgr
):

    x = preprocess_frame(
        frame_bgr
    )


    start = time.perf_counter()


    result = compiled_model(
        [x]
    )


    elapsed_ms = (
        time.perf_counter()
        - start
    ) * 1000.0


    embedding = np.asarray(
        result[output_layer]
    ).reshape(-1)


    if embedding.shape != (960,):

        raise ValueError(
            "Unexpected frame embedding "
            f"shape: {embedding.shape}"
        )


    return (
        embedding.astype(np.float32),
        float(elapsed_ms),
    )


# ==================================================================================================
# 9. VIDEO EMBEDDING
# ==================================================================================================

def extract_video_embedding(
    video_path
):

    video_path = Path(
        video_path
    )


    info = inspect_video(
        video_path
    )


    indices = generate_uniform_indices(
        frame_count=
            info["frame_count"],

        num_frames=
            TEMPORAL_FRAMES
    )


    frames = read_sampled_frames(
        video_path,
        indices
    )


    frame_embeddings = []
    inference_times_ms = []


    total_start = time.perf_counter()


    for frame in frames:

        embedding, latency_ms = (
            extract_frame_embedding(
                frame
            )
        )

        frame_embeddings.append(
            embedding
        )

        inference_times_ms.append(
            latency_ms
        )


    frame_embeddings = np.stack(
        frame_embeddings,
        axis=0
    )


    # Frozen temporal aggregation
    video_embedding = (
        frame_embeddings.mean(
            axis=0
        )
    )


    norm = float(
        np.linalg.norm(
            video_embedding
        )
    )


    if norm <= 1e-12:

        raise RuntimeError(
            "Video embedding has zero norm."
        )


    video_embedding = (
        video_embedding
        / norm
    ).astype(np.float32)


    total_elapsed_ms = (
        time.perf_counter()
        - total_start
    ) * 1000.0


    diagnostics = {

        "video_path":
            str(video_path),

        "frame_count":
            int(
                info["frame_count"]
            ),

        "fps":
            float(
                info["fps"]
            ),

        "width":
            int(
                info["width"]
            ),

        "height":
            int(
                info["height"]
            ),

        "duration_seconds":
            (
                float(
                    info["duration_seconds"]
                )
                if info["duration_seconds"]
                is not None
                else None
            ),

        "sampled_indices":
            [
                int(i)
                for i in indices
            ],

        "frame_embedding_shape":
            list(
                frame_embeddings.shape
            ),

        "video_embedding_shape":
            list(
                video_embedding.shape
            ),

        "video_embedding_norm":
            float(
                np.linalg.norm(
                    video_embedding
                )
            ),

        "mean_inference_ms":
            float(
                np.mean(
                    inference_times_ms
                )
            ),

        "median_inference_ms":
            float(
                np.median(
                    inference_times_ms
                )
            ),

        "p95_inference_ms":
            float(
                np.percentile(
                    inference_times_ms,
                    95
                )
            ),

        "total_embedding_pipeline_ms":
            float(
                total_elapsed_ms
            ),
    }


    return (
        video_embedding,
        diagnostics
    )


# ==================================================================================================
# 10. RETRIEVAL
# ==================================================================================================

def retrieve_topk(
    query_embedding,
    k=5
):

    retrieval_start = (
        time.perf_counter()
    )


    similarities = (
        prototype_bank
        @ query_embedding
    )


    k = min(
        int(k),
        len(similarities)
    )


    top_indices = np.argpartition(
        -similarities,
        k - 1
    )[:k]


    top_indices = top_indices[
        np.argsort(
            -similarities[
                top_indices
            ]
        )
    ]


    retrieval_ms = (
        time.perf_counter()
        - retrieval_start
    ) * 1000.0


    results = []


    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        row = prototype_meta.iloc[
            int(idx)
        ]


        results.append(
            {
                "rank":
                    int(rank),

                "prototype_index":
                    int(idx),

                "prototype_uid":
                    str(
                        row["uid"]
                    ),

                "gloss":
                    str(
                        row["gloss"]
                    ),

                "similarity":
                    float(
                        similarities[idx]
                    ),
            }
        )


    return (
        results,
        float(retrieval_ms)
    )


# ==================================================================================================
# 11. COMPLETE OFFLINE RECOGNIZER
# ==================================================================================================

def recognize_video(
    video_path,
    expected_gloss=None,
    verbose=True
):

    video_path = Path(
        video_path
    )


    if not video_path.exists():

        raise FileNotFoundError(
            f"Video does not exist: "
            f"{video_path}"
        )


    complete_start = (
        time.perf_counter()
    )


    query_embedding, diagnostics = (
        extract_video_embedding(
            video_path
        )
    )


    predictions, retrieval_ms = (
        retrieve_topk(
            query_embedding,
            k=TOP_K
        )
    )


    end_to_end_ms = (
        time.perf_counter()
        - complete_start
    ) * 1000.0


    top1 = predictions[0]


    result = {

        "video_path":
            str(video_path),

        "expected_gloss":
            (
                str(expected_gloss)
                if expected_gloss
                is not None
                else None
            ),

        "predicted_gloss":
            top1["gloss"],

        "top1_similarity":
            float(
                top1["similarity"]
            ),

        "top1_correct":
            (
                bool(
                    top1["gloss"]
                    == str(
                        expected_gloss
                    )
                )
                if expected_gloss
                is not None
                else None
            ),

        "top5_contains_expected":
            (
                bool(
                    str(
                        expected_gloss
                    )
                    in [
                        p["gloss"]
                        for p in predictions
                    ]
                )
                if expected_gloss
                is not None
                else None
            ),

        "predictions":
            predictions,

        "retrieval_ms":
            float(
                retrieval_ms
            ),

        "end_to_end_ms":
            float(
                end_to_end_ms
            ),

        "diagnostics":
            diagnostics,
    }


    if verbose:

        print("\n" + "=" * 100)
        print("OFFLINE RECOGNITION")
        print("=" * 100)

        print(
            "Video:",
            video_path
        )

        print(
            "Frames in video:",
            diagnostics[
                "frame_count"
            ]
        )

        print(
            "FPS:",
            f"{diagnostics['fps']:.3f}"
        )

        print(
            "Duration:",
            (
                f"{diagnostics['duration_seconds']:.3f} s"
                if diagnostics[
                    "duration_seconds"
                ] is not None
                else "unknown"
            )
        )

        print(
            "Automatically sampled:",
            diagnostics[
                "sampled_indices"
            ]
        )


        if expected_gloss is not None:

            print(
                "\nExpected gloss:",
                expected_gloss
            )


        print("\nTOP-5 PREDICTIONS")
        print("-" * 100)


        for pred in predictions:

            marker = ""

            if (
                expected_gloss is not None
                and pred["gloss"]
                == str(
                    expected_gloss
                )
            ):

                marker = " <-- TRUE"


            print(
                f"{pred['rank']:>2}. "
                f"{pred['gloss']:<35} "
                f"{pred['similarity']:.6f}"
                f"{marker}"
            )


        print("\nTIMING")
        print("-" * 100)

        print(
            "Mean OV frame inference : "
            f"{diagnostics['mean_inference_ms']:.3f} ms"
        )

        print(
            "P95 OV frame inference  : "
            f"{diagnostics['p95_inference_ms']:.3f} ms"
        )

        print(
            "Retrieval               : "
            f"{retrieval_ms:.3f} ms"
        )

        print(
            "End-to-end function     : "
            f"{end_to_end_ms:.3f} ms"
        )


    return result


# ==================================================================================================
# 12. SELECT A SMALL, DETERMINISTIC VALIDATION SET
# ==================================================================================================
#
# We deliberately use several examples rather than selecting only known-correct
# signs. This prevents the Stage 9.4 demo from becoming cherry-picked.
#
# The label is used ONLY after prediction for reporting.
#
# ==================================================================================================

if "status" in test_meta.columns:

    valid_test = test_meta[
        test_meta["status"]
        .astype(str)
        .str.upper()
        == "OK"
    ].copy()

else:

    valid_test = (
        test_meta.copy()
    )


N_VALIDATION = 10


validation_rows = (
    valid_test
    .head(N_VALIDATION)
    .copy()
)


print("\n" + "=" * 100)
print("STAGE 9.4 VALIDATION VIDEOS")
print("=" * 100)


print(
    validation_rows[
        [
            "uid",
            "gloss",
            "video_path"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 13. RUN OFFLINE LIVE-STYLE VALIDATION
# ==================================================================================================

all_results = []
top5_rows = []


for sample_number, (_, row) in enumerate(
    validation_rows.iterrows(),
    start=1
):

    uid = str(
        row["uid"]
    )

    expected_gloss = str(
        row["gloss"]
    )

    video_path = Path(
        str(
            row["video_path"]
        )
    )


    print(
        "\n\n"
        + "#" * 100
    )

    print(
        f"VIDEO "
        f"{sample_number}/"
        f"{len(validation_rows)}"
    )

    print(
        f"UID   : {uid}"
    )

    print(
        f"GLOSS : {expected_gloss}"
    )

    print(
        "#" * 100
    )


    try:

        result = recognize_video(
            video_path=
                video_path,

            expected_gloss=
                expected_gloss,

            verbose=True
        )


        diagnostics = (
            result[
                "diagnostics"
            ]
        )


        all_results.append(
            {
                "uid":
                    uid,

                "expected_gloss":
                    expected_gloss,

                "video_path":
                    str(
                        video_path
                    ),

                "status":
                    "OK",

                "predicted_gloss":
                    result[
                        "predicted_gloss"
                    ],

                "top1_similarity":
                    result[
                        "top1_similarity"
                    ],

                "top1_correct":
                    result[
                        "top1_correct"
                    ],

                "top5_contains_expected":
                    result[
                        "top5_contains_expected"
                    ],

                "frame_count":
                    diagnostics[
                        "frame_count"
                    ],

                "video_fps":
                    diagnostics[
                        "fps"
                    ],

                "sampled_indices":
                    json.dumps(
                        diagnostics[
                            "sampled_indices"
                        ]
                    ),

                "mean_inference_ms":
                    diagnostics[
                        "mean_inference_ms"
                    ],

                "p95_inference_ms":
                    diagnostics[
                        "p95_inference_ms"
                    ],

                "retrieval_ms":
                    result[
                        "retrieval_ms"
                    ],

                "end_to_end_ms":
                    result[
                        "end_to_end_ms"
                    ],
            }
        )


        for pred in result[
            "predictions"
        ]:

            top5_rows.append(
                {
                    "query_uid":
                        uid,

                    "expected_gloss":
                        expected_gloss,

                    "rank":
                        pred[
                            "rank"
                        ],

                    "prototype_uid":
                        pred[
                            "prototype_uid"
                        ],

                    "predicted_gloss":
                        pred[
                            "gloss"
                        ],

                    "similarity":
                        pred[
                            "similarity"
                        ],

                    "is_expected":
                        pred[
                            "gloss"
                        ]
                        == expected_gloss,
                }
            )


    except Exception as exc:

        print(
            "\n[ERROR]",
            type(exc).__name__,
            str(exc)
        )


        all_results.append(
            {
                "uid":
                    uid,

                "expected_gloss":
                    expected_gloss,

                "video_path":
                    str(
                        video_path
                    ),

                "status":
                    (
                        f"ERROR: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            }
        )


# ==================================================================================================
# 14. RESULTS TABLE
# ==================================================================================================

results_df = pd.DataFrame(
    all_results
)

top5_df = pd.DataFrame(
    top5_rows
)


print("\n" + "=" * 100)
print("STAGE 9.4 RESULTS")
print("=" * 100)


print(
    results_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 15. SUMMARY METRICS
# ==================================================================================================

ok_df = results_df[
    results_df["status"]
    == "OK"
].copy()


if len(ok_df) == 0:

    raise RuntimeError(
        "No Stage 9.4 videos "
        "were successfully evaluated."
    )


top1_sample_accuracy = float(
    ok_df[
        "top1_correct"
    ].mean()
)

top5_sample_accuracy = float(
    ok_df[
        "top5_contains_expected"
    ].mean()
)

mean_inference_ms = float(
    ok_df[
        "mean_inference_ms"
    ].mean()
)

mean_retrieval_ms = float(
    ok_df[
        "retrieval_ms"
    ].mean()
)

mean_end_to_end_ms = float(
    ok_df[
        "end_to_end_ms"
    ].mean()
)


print("\n" + "-" * 100)
print("SUMMARY")
print("-" * 100)


print(
    "Videos requested          :",
    len(validation_rows)
)

print(
    "Videos successfully tested:",
    len(ok_df)
)

print(
    "Sample Top-1 accuracy     :",
    f"{top1_sample_accuracy:.4f}"
)

print(
    "Sample Top-5 accuracy     :",
    f"{top5_sample_accuracy:.4f}"
)

print(
    "Mean OV frame inference   :",
    f"{mean_inference_ms:.3f} ms"
)

print(
    "Mean retrieval latency    :",
    f"{mean_retrieval_ms:.3f} ms"
)

print(
    "Mean end-to-end function  :",
    f"{mean_end_to_end_ms:.3f} ms"
)


# ==================================================================================================
# 16. SAVE RESULTS
# ==================================================================================================

RESULTS_CSV = (
    PREDICTION_DIR
    / "stage9_4_offline_validation.csv"
)

TOP5_CSV = (
    PREDICTION_DIR
    / "stage9_4_top5_predictions.csv"
)

REPORT_JSON = (
    REPORT_DIR
    / "stage9_4_offline_validation_summary.json"
)


results_df.to_csv(
    RESULTS_CSV,
    index=False
)

top5_df.to_csv(
    TOP5_CSV,
    index=False
)


report = {

    "stage":
        "9.4",

    "name":
        "Offline Live-Style Video Recognition Validation",

    "status":
        (
            "PASS"
            if len(ok_df)
            == len(validation_rows)
            else "COMPLETE_WITH_ERRORS"
        ),

    "openvino_device":
        OPENVINO_DEVICE,

    "openvino_model":
        str(
            OV_XML
        ),

    "prototype_bank":
        str(
            PROTOTYPE_EMBED_PATH
        ),

    "prototype_count":
        int(
            len(prototype_bank)
        ),

    "embedding_dimension":
        int(
            prototype_bank.shape[1]
        ),

    "temporal_frames":
        int(
            TEMPORAL_FRAMES
        ),

    "sampling":
        "automatic uniform full-video sampling",

    "historical_sample_indices_used":
        False,

    "videos_requested":
        int(
            len(validation_rows)
        ),

    "videos_successfully_tested":
        int(
            len(ok_df)
        ),

    "sample_top1_accuracy":
        top1_sample_accuracy,

    "sample_top5_accuracy":
        top5_sample_accuracy,

    "mean_openvino_frame_inference_ms":
        mean_inference_ms,

    "mean_retrieval_latency_ms":
        mean_retrieval_ms,

    "mean_end_to_end_function_ms":
        mean_end_to_end_ms,

    "methodology": {

        "input":
            "complete isolated-sign video",

        "frame_sampling":
            "8 frames uniformly sampled automatically",

        "spatial_input":
            "full frame",

        "spatial_model":
            "OpenVINO MobileNetV3-Large 960-D extractor",

        "temporal_aggregation":
            "mean pooling",

        "normalization":
            "L2",

        "retrieval":
            "cosine similarity",

        "gallery":
            "frozen Stage 4.5 canonical prototype gallery",

        "hand_roi":
            False,

        "tta":
            False,

        "whitening":
            False,

        "csls":
            False,

        "test_label_tuning":
            False,
    },

    "scientific_note":
        (
            "Sample accuracy from this small deployment-validation "
            "subset must not replace the frozen official Stage 4.5 "
            "evaluation metrics."
        ),
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


print("\nSaved:")
print(RESULTS_CSV)
print(TOP5_CSV)
print(REPORT_JSON)


# ==================================================================================================
# 17. STAGE DECISION
# ==================================================================================================

print("\n" + "=" * 100)


if len(ok_df) == len(
    validation_rows
):

    print(
        "STAGE 9.4 — PASS"
    )

    print(
        "\nAll requested prerecorded "
        "CISLR videos were processed "
        "without using historical "
        "sample indices."
    )

    print(
        "\nThe deployment engine can now "
        "progress to Stage 9.5 live "
        "webcam inference."
    )

else:

    print(
        "STAGE 9.4 — "
        "COMPLETE WITH ERRORS"
    )

    print(
        "\nDo not proceed to webcam "
        "integration until the failed "
        "videos are inspected."
    )


print(
    "\nIMPORTANT:"
)

print(
    "The sample accuracy above is a "
    "deployment diagnostic, not the "
    "official CISLR test accuracy."
)

print("=" * 100)

CISLR — STAGE 9.4
OFFLINE LIVE-STYLE VIDEO RECOGNITION VALIDATION

ARTIFACT CHECK
----------------------------------------------------------------------------------------------------
OpenVINO model           : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage9_final_openvino/embedding_fp32/mobilenet_v3_large_embedding_960_fp32_static.xml
                           Exists = True
Prototype embeddings     : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_embeddings.npy
                           Exists = True
Prototype metadata       : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_metadata.csv
                           Exists = True
Test metadata            : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_test_metadata.csv
                           Exists = True

[OK] All required artifacts ex

In [2]:
# ==================================================================================================
# CISLR — STAGE 9.4.1
# ROBUST DECODE CORRECTION FOR OFFLINE LIVE-STYLE RECOGNITION
# ==================================================================================================
#
# PURPOSE
# -------
# Correct the Stage 9.4 dependency on container-reported frame counts.
#
# Instead of:
#   reported frame count -> random seek -> decode
#
# use:
#   sequential decode -> actual readable frames -> uniform sampling
#
# NO MODEL CHANGE.
# NO RETRAINING.
# NO PROTOTYPE CHANGE.
# NO TEST-LABEL TUNING.
#
# ==================================================================================================

from pathlib import Path
import json
import time

import cv2
import numpy as np
import pandas as pd


print("=" * 100)
print("CISLR — STAGE 9.4.1")
print("ROBUST DECODE CORRECTION")
print("=" * 100)


# ==================================================================================================
# 1. ROBUST SEQUENTIAL VIDEO DECODER
# ==================================================================================================

def decode_all_readable_frames(video_path):
    """
    Sequentially decode every frame OpenCV can actually read.

    This deliberately does NOT trust CAP_PROP_FRAME_COUNT for sampling.
    """

    video_path = Path(video_path)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    reported_frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    frames = []

    while True:

        ok, frame = cap.read()

        if not ok or frame is None:
            break

        frames.append(frame)

    cap.release()

    actual_decoded_count = len(frames)

    if actual_decoded_count == 0:
        raise RuntimeError(
            f"No readable frames found in: {video_path}"
        )

    return frames, {
        "reported_frame_count": reported_frame_count,
        "actual_decoded_count": actual_decoded_count,
        "fps": fps,
        "width": width,
        "height": height,
    }


# ==================================================================================================
# 2. SAMPLE FROM ACTUALLY DECODED FRAMES
# ==================================================================================================

def sample_from_decoded_frames(
    decoded_frames,
    num_frames=8
):

    n = len(decoded_frames)

    if n == 0:
        raise RuntimeError(
            "Decoded-frame list is empty."
        )

    indices = np.linspace(
        0,
        n - 1,
        num_frames
    )

    indices = np.rint(
        indices
    ).astype(int)

    indices = np.clip(
        indices,
        0,
        n - 1
    )

    sampled_frames = [
        decoded_frames[int(i)]
        for i in indices
    ]

    return sampled_frames, indices.tolist()


# ==================================================================================================
# 3. CORRECTED VIDEO EMBEDDING FUNCTION
# ==================================================================================================

def extract_video_embedding_robust(
    video_path
):

    total_start = time.perf_counter()

    decoded_frames, video_info = (
        decode_all_readable_frames(
            video_path
        )
    )

    sampled_frames, sampled_indices = (
        sample_from_decoded_frames(
            decoded_frames,
            TEMPORAL_FRAMES
        )
    )

    frame_embeddings = []
    inference_times_ms = []

    for frame in sampled_frames:

        embedding, latency_ms = (
            extract_frame_embedding(
                frame
            )
        )

        frame_embeddings.append(
            embedding
        )

        inference_times_ms.append(
            latency_ms
        )

    frame_embeddings = np.stack(
        frame_embeddings,
        axis=0
    )

    video_embedding = (
        frame_embeddings.mean(axis=0)
    )

    norm = float(
        np.linalg.norm(
            video_embedding
        )
    )

    if norm <= 1e-12:
        raise RuntimeError(
            "Video embedding has zero norm."
        )

    video_embedding = (
        video_embedding / norm
    ).astype(np.float32)

    total_ms = (
        time.perf_counter()
        - total_start
    ) * 1000.0

    diagnostics = {

        **video_info,

        "sampled_indices":
            [int(i) for i in sampled_indices],

        "frame_embedding_shape":
            list(frame_embeddings.shape),

        "video_embedding_shape":
            list(video_embedding.shape),

        "video_embedding_norm":
            float(np.linalg.norm(video_embedding)),

        "mean_inference_ms":
            float(np.mean(inference_times_ms)),

        "median_inference_ms":
            float(np.median(inference_times_ms)),

        "p95_inference_ms":
            float(
                np.percentile(
                    inference_times_ms,
                    95
                )
            ),

        "total_pipeline_ms":
            float(total_ms),
    }

    return video_embedding, diagnostics


# ==================================================================================================
# 4. CORRECTED RECOGNITION FUNCTION
# ==================================================================================================

def recognize_video_robust(
    video_path,
    expected_gloss=None,
    verbose=True
):

    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(
            f"Video does not exist: {video_path}"
        )

    start = time.perf_counter()

    query_embedding, diagnostics = (
        extract_video_embedding_robust(
            video_path
        )
    )

    predictions, retrieval_ms = (
        retrieve_topk(
            query_embedding,
            TOP_K
        )
    )

    end_to_end_ms = (
        time.perf_counter() - start
    ) * 1000.0

    predicted_gloss = (
        predictions[0]["gloss"]
    )

    top5_glosses = [
        p["gloss"]
        for p in predictions
    ]

    result = {

        "video_path":
            str(video_path),

        "expected_gloss":
            expected_gloss,

        "predicted_gloss":
            predicted_gloss,

        "top1_similarity":
            float(
                predictions[0]["similarity"]
            ),

        "top1_correct":
            (
                predicted_gloss
                == expected_gloss
                if expected_gloss is not None
                else None
            ),

        "top5_contains_expected":
            (
                expected_gloss in top5_glosses
                if expected_gloss is not None
                else None
            ),

        "predictions":
            predictions,

        "retrieval_ms":
            float(retrieval_ms),

        "end_to_end_ms":
            float(end_to_end_ms),

        "diagnostics":
            diagnostics,
    }

    if verbose:

        print("\n" + "=" * 100)
        print("ROBUST OFFLINE RECOGNITION")
        print("=" * 100)

        print("Video:", video_path)

        print(
            "Reported frame count :",
            diagnostics[
                "reported_frame_count"
            ]
        )

        print(
            "Actually decoded     :",
            diagnostics[
                "actual_decoded_count"
            ]
        )

        print(
            "Sampled indices      :",
            diagnostics[
                "sampled_indices"
            ]
        )

        if expected_gloss is not None:
            print(
                "Expected gloss       :",
                expected_gloss
            )

        print("\nTOP-5")
        print("-" * 100)

        for pred in predictions:

            marker = ""

            if (
                expected_gloss is not None
                and pred["gloss"]
                == expected_gloss
            ):
                marker = " <-- TRUE"

            print(
                f"{pred['rank']:>2}. "
                f"{pred['gloss']:<35} "
                f"{pred['similarity']:.6f}"
                f"{marker}"
            )

        print("\nTIMING")
        print("-" * 100)

        print(
            "Mean OV inference : "
            f"{diagnostics['mean_inference_ms']:.3f} ms"
        )

        print(
            "P95 OV inference  : "
            f"{diagnostics['p95_inference_ms']:.3f} ms"
        )

        print(
            "Retrieval         : "
            f"{retrieval_ms:.3f} ms"
        )

        print(
            "End-to-end        : "
            f"{end_to_end_ms:.3f} ms"
        )

    return result


# ==================================================================================================
# 5. RETEST THE SAME 10 VIDEOS
# ==================================================================================================

corrected_rows = []
corrected_top5_rows = []


for number, (_, row) in enumerate(
    validation_rows.iterrows(),
    start=1
):

    uid = str(row["uid"])
    expected_gloss = str(row["gloss"])
    video_path = Path(str(row["video_path"]))

    print("\n\n" + "#" * 100)
    print(
        f"CORRECTED VIDEO "
        f"{number}/{len(validation_rows)}"
    )
    print("UID   :", uid)
    print("GLOSS :", expected_gloss)
    print("#" * 100)

    try:

        result = recognize_video_robust(
            video_path,
            expected_gloss=expected_gloss,
            verbose=True
        )

        d = result["diagnostics"]

        corrected_rows.append(
            {
                "uid":
                    uid,

                "expected_gloss":
                    expected_gloss,

                "status":
                    "OK",

                "reported_frame_count":
                    d["reported_frame_count"],

                "actual_decoded_count":
                    d["actual_decoded_count"],

                "sampled_indices":
                    json.dumps(
                        d["sampled_indices"]
                    ),

                "predicted_gloss":
                    result[
                        "predicted_gloss"
                    ],

                "top1_similarity":
                    result[
                        "top1_similarity"
                    ],

                "top1_correct":
                    result[
                        "top1_correct"
                    ],

                "top5_contains_expected":
                    result[
                        "top5_contains_expected"
                    ],

                "mean_inference_ms":
                    d[
                        "mean_inference_ms"
                    ],

                "p95_inference_ms":
                    d[
                        "p95_inference_ms"
                    ],

                "retrieval_ms":
                    result[
                        "retrieval_ms"
                    ],

                "end_to_end_ms":
                    result[
                        "end_to_end_ms"
                    ],
            }
        )

        for pred in result["predictions"]:

            corrected_top5_rows.append(
                {
                    "query_uid":
                        uid,

                    "expected_gloss":
                        expected_gloss,

                    "rank":
                        pred["rank"],

                    "prototype_uid":
                        pred[
                            "prototype_uid"
                        ],

                    "predicted_gloss":
                        pred["gloss"],

                    "similarity":
                        pred[
                            "similarity"
                        ],

                    "is_expected":
                        pred["gloss"]
                        == expected_gloss,
                }
            )

    except Exception as exc:

        print(
            "[ERROR]",
            type(exc).__name__,
            str(exc)
        )

        corrected_rows.append(
            {
                "uid":
                    uid,

                "expected_gloss":
                    expected_gloss,

                "status":
                    (
                        f"ERROR: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            }
        )


# ==================================================================================================
# 6. RESULTS
# ==================================================================================================

corrected_df = pd.DataFrame(
    corrected_rows
)

corrected_top5_df = pd.DataFrame(
    corrected_top5_rows
)


print("\n" + "=" * 100)
print("STAGE 9.4.1 RESULTS")
print("=" * 100)

print(
    corrected_df.to_string(
        index=False
    )
)


ok_corrected = corrected_df[
    corrected_df["status"] == "OK"
].copy()


if len(ok_corrected) == 0:
    raise RuntimeError(
        "No videos were successfully processed."
    )


top1_accuracy = float(
    ok_corrected[
        "top1_correct"
    ].mean()
)

top5_accuracy = float(
    ok_corrected[
        "top5_contains_expected"
    ].mean()
)

mean_inference = float(
    ok_corrected[
        "mean_inference_ms"
    ].mean()
)

mean_retrieval = float(
    ok_corrected[
        "retrieval_ms"
    ].mean()
)

mean_end_to_end = float(
    ok_corrected[
        "end_to_end_ms"
    ].mean()
)


print("\n" + "-" * 100)
print("SUMMARY")
print("-" * 100)

print(
    "Videos requested          :",
    len(validation_rows)
)

print(
    "Videos successfully tested:",
    len(ok_corrected)
)

print(
    "Sample Top-1 accuracy     :",
    f"{top1_accuracy:.4f}"
)

print(
    "Sample Top-5 accuracy     :",
    f"{top5_accuracy:.4f}"
)

print(
    "Mean OV frame inference   :",
    f"{mean_inference:.3f} ms"
)

print(
    "Mean retrieval latency    :",
    f"{mean_retrieval:.3f} ms"
)

print(
    "Mean end-to-end latency   :",
    f"{mean_end_to_end:.3f} ms"
)


# ==================================================================================================
# 7. SAVE CORRECTED RESULTS
# ==================================================================================================

CORRECTED_RESULTS_CSV = (
    PREDICTION_DIR
    / "stage9_4_1_robust_offline_validation.csv"
)

CORRECTED_TOP5_CSV = (
    PREDICTION_DIR
    / "stage9_4_1_robust_top5_predictions.csv"
)

CORRECTED_REPORT_JSON = (
    REPORT_DIR
    / "stage9_4_1_robust_decode_summary.json"
)


corrected_df.to_csv(
    CORRECTED_RESULTS_CSV,
    index=False
)

corrected_top5_df.to_csv(
    CORRECTED_TOP5_CSV,
    index=False
)


report = {

    "stage":
        "9.4.1",

    "name":
        "Robust Decode Correction",

    "reason":
        (
            "Stage 9.4 exposed a mismatch "
            "between container-reported frame "
            "count and actually decodable frames."
        ),

    "model_changed":
        False,

    "prototype_gallery_changed":
        False,

    "retrieval_method_changed":
        False,

    "input_pipeline_changed":
        True,

    "input_pipeline_change":
        (
            "Sequentially decode readable frames "
            "before automatic temporal sampling."
        ),

    "videos_requested":
        int(len(validation_rows)),

    "videos_successfully_tested":
        int(len(ok_corrected)),

    "sample_top1_accuracy":
        top1_accuracy,

    "sample_top5_accuracy":
        top5_accuracy,

    "mean_openvino_frame_inference_ms":
        mean_inference,

    "mean_retrieval_latency_ms":
        mean_retrieval,

    "mean_end_to_end_latency_ms":
        mean_end_to_end,
}


with open(
    CORRECTED_REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


print("\nSaved:")
print(CORRECTED_RESULTS_CSV)
print(CORRECTED_TOP5_CSV)
print(CORRECTED_REPORT_JSON)


# ==================================================================================================
# 8. FINAL DECISION
# ==================================================================================================

print("\n" + "=" * 100)

if len(ok_corrected) == len(
    validation_rows
):

    print(
        "STAGE 9.4.1 — PASS"
    )

    print(
        "\nAll validation videos were "
        "successfully processed using "
        "actual decodable frames."
    )

    print(
        "\nThe robust sequential decoding "
        "strategy should be retained for "
        "offline arbitrary-video inference."
    )

    print(
        "\nREADY FOR STAGE 9.5 — "
        "LIVE WEBCAM INFERENCE"
    )

else:

    print(
        "STAGE 9.4.1 — "
        "COMPLETE WITH ERRORS"
    )

    print(
        "\nDo NOT proceed to webcam yet."
    )

print("=" * 100)

CISLR — STAGE 9.4.1
ROBUST DECODE CORRECTION


####################################################################################################
CORRECTED VIDEO 1/10
UID   : -NIz596Y27Q_1
GLOSS : severally
####################################################################################################

ROBUST OFFLINE RECOGNITION
Video: /home/dipshikha/Music/cao/CISLR_v1.5-a_videos/-NIz596Y27Q_1.mp4
Reported frame count : 45
Actually decoded     : 45
Sampled indices      : [0, 6, 13, 19, 25, 31, 38, 44]
Expected gloss       : severally

TOP-5
----------------------------------------------------------------------------------------------------
 1. sample                              0.934726
 2. salary                              0.923429
 3. semi urban                          0.920906
 4. scope                               0.918661
 5. review                              0.915093

TIMING
-------------------------------------------------------------------------------------------